In [12]:
import os
import torch
import torch.utils.data
import torchvision
from PIL import Image
from pycocotools.coco import COCO
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import FasterRCNN_ResNet50_FPN_Weights
from torchvision.transforms import functional as F
import sys
import time
import json

# ----------------------------
# 1. Configuration & Engine Utils (Condensed)
# ----------------------------

# Helper class to keep track of average values
class Averager:
    def __init__(self):
        self.current_total = 0.0
        self.iterations = 0.0

    def send(self, value):
        self.current_total += value
        self.iterations += 1

    @property
    def value(self):
        if self.iterations == 0:
            return 0
        else:
            return 1.0 * self.current_total / self.iterations

    def reset(self):
        self.current_total = 0.0
        self.iterations = 0.0

# Logger to write to both console and file
class DualLogger(object):
    def __init__(self, filepath):
        self.terminal = sys.stdout
        self.log = open(filepath, "w")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        # this flush method is needed for python 3 compatibility.
        # this handles the flush command by doing nothing.
        # you might want to specify some extra behavior here.
        self.terminal.flush()
        self.log.flush()

# ----------------------------
# 2. Dataset Definition
# ----------------------------

class CastingDataset(torch.utils.data.Dataset):
    def __init__(self, root, annotation_file, transforms=None):
        self.root = root
        self.transforms = transforms
        self.coco = COCO(annotation_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        
        # Load categories to map IDs to names later
        self.cats = self.coco.loadCats(self.coco.getCatIds())
        self.cat_id_to_name = {cat['id']: cat['name'] for cat in self.cats}
        
        # Filter out images without annotations if necessary, 
        # but for object detection it's okay to have empty images (negatives).
        # We keep all provided IDs.

    def __getitem__(self, index):
        # COCO Image ID
        coco_id = self.ids[index]
        img_metadata = self.coco.loadImgs(coco_id)[0]
        path = img_metadata['file_name']
        
        # Open Image
        img_path = os.path.join(self.root, path)
        img = Image.open(img_path).convert("RGB")
        
        # Get Annotations
        ann_ids = self.coco.getAnnIds(imgIds=coco_id)
        coco_anns = self.coco.loadAnns(ann_ids)
        
        boxes = []
        labels = []
        areas = []
        iscrowd = []

        num_objs = len(coco_anns)
        
        for i in range(num_objs):
            # COCO bbox format is [xmin, ymin, width, height]
            xmin = coco_anns[i]['bbox'][0]
            ymin = coco_anns[i]['bbox'][1]
            w = coco_anns[i]['bbox'][2]
            h = coco_anns[i]['bbox'][3]
            
            # Convert to [xmin, ymin, xmax, ymax]
            xmax = xmin + w
            ymax = ymin + h
            
            # Handle empty boxes/degenerate boxes
            if w <= 0 or h <= 0:
                continue

            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(coco_anns[i]['category_id']) # Ensure your category IDs match model expectations
            areas.append(coco_anns[i]['area'])
            iscrowd.append(coco_anns[i]['iscrowd'])

        # Convert to tensors
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            areas = torch.as_tensor(areas, dtype=torch.float32)
            iscrowd = torch.as_tensor(iscrowd, dtype=torch.int64)
        else:
            # Handle images with no defects (negative samples)
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        image_id = torch.tensor([coco_id])
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = image_id
        target["area"] = areas
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.ids)

def get_transform():
    # Simple transform: convert to tensor
    # You can add random flips, normalization, etc. here
    def transform(image):
        image = F.to_tensor(image)
        return image
    return transform

# ----------------------------
# 3. Model Helper
# ----------------------------

def get_model_instance_segmentation(num_classes):
    # Load an instance segmentation model pre-trained on COCO
    # We use Faster R-CNN with ResNet-50 FPN
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None)

    # Get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# ----------------------------
# 4. Training & Evaluation Functions
# ----------------------------

def train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10):
    model.train()
    metric_logger = Averager()
    
    header = 'Epoch: [{}]'.format(epoch)
    
    start_time = time.time()
    
    iter_loss = 0.0
    
    for i, (images, targets) in enumerate(data_loader):
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        # Reduce losses over all GPUs for logging purposes if necessary, 
        # but here we assume single GPU for Colab.
        loss_value = losses.item()

        if not os.path.isfile("loss_log.txt"): # initialize
             with open("loss_log.txt", "w") as f:
                 f.write("Epoch,Iteration,Loss\n")
        
        # Log granular loss step
        with open("loss_log.txt", "a") as f:
            f.write(f"{epoch},{i},{loss_value:.4f}\n")

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        iter_loss += loss_value

        if i % print_freq == 0:
            print(f"{header} Iter: {i} Loss: {loss_value:.4f}")
            
    print(f"{header} Average Loss: {iter_loss / len(data_loader):.4f}")

@torch.no_grad()
def evaluate(model, data_loader, device, num_classes, cat_id_to_name):
    """
    Manual implementation of evaluation to calculate class-wise metrics easily 
    without relying obscure engine imports.
    Note: For strict COCO mAP, it is best to use Cocoevaluator from references, 
    but this gives us direct control over output formatting.
    """
    model.eval()
    
    print("\n--- Starting Evaluation ---")
    
    # Using pycocotools for official metrics
    from pycocotools.cocoeval import COCOeval
    
    cpu_device = torch.device("cpu")
    
    # Needs to convert model outputs to COCO results format
    coco = data_loader.dataset.coco
    results = []
    
    for images, targets in data_loader:
        images = list(img.to(device) for img in images)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            
        outputs = model(images)
        outputs = [{k: v.to(cpu_device) for k, v in t.items()} for t in outputs]
        
        for target, output in zip(targets, outputs):
            image_id = target["image_id"].item()
            
            # Format outputs for COCOEval
            boxes = output["boxes"].tolist()
            scores = output["scores"].tolist()
            labels = output["labels"].tolist()
            
            for box, score, label in zip(boxes, scores, labels):
                # Convert back to x, y, w, h
                res = {
                    "image_id": image_id,
                    "category_id": label,
                    "bbox": [box[0], box[1], box[2] - box[0], box[3] - box[1]],
                    "score": score
                }
                results.append(res)

    if not results:
        print("No detections generated! Check model or data.")
        return

    # Write detections to file
    with open("val_predictions.json", "w") as f:
        json.dump(results, f)
        
    coco_dt = coco.loadRes(results)
    
    # Run COCO Evaluation
    coco_eval = COCOeval(coco, coco_dt, iouType='bbox')
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    
    # Extract and Print Class-wise Metrics
    print("\n--- Class-wise Metrics (AP @ IoU=0.50:0.95) ---")
    precisions = coco_eval.eval['precision']
    # precision shape: [TxRxKxAxM]
    # T: iou thresholds (10)
    # R: recall thresholds (101)
    # K: category ids (num_classes)
    # A: area ranges (4)
    # M: max dets (3)
    
    class_metrics = {}
    
    # We want mean over Iou(T), Recall(R), Area(0=all), MaxDets(2=100)
    for cat_id in cat_id_to_name:
        # Values are stored by index in coco_eval.params.catIds
        cat_idx = -1
        if cat_id in coco_eval.params.catIds:
            cat_idx = coco_eval.params.catIds.index(cat_id)
        
        if cat_idx != -1:
            # Average over IoU thresholds (axis 0) and recall thresholds (axis 1)
            # indices: [:, :, cat_idx, 0, 2] -> 0 is 'all' area, 2 is max_dets=100
            p = precisions[:, :, cat_idx, 0, 2]
            ap = p.mean() # Mean Average Precision for this class
            
            cat_name = cat_id_to_name[cat_id]
            print(f"Class '{cat_name}' (ID {cat_id}): AP = {ap:.4f}")
            class_metrics[cat_name] = float(ap)
            
    # Save metrics to file
    with open("metrics_results.json", "w") as f:
        json.dump(class_metrics, f, indent=4)
        
    print("\nMetrics saved to metrics_results.json")

# ----------------------------
# 5. Visualization Helper
# ----------------------------

def visualize_prediction(image_path, model, device, threshold=0.5, save_path="prediction.jpg"):
    model.eval()
    img = Image.open(image_path).convert("RGB")
    transform = get_transform()
    img_tensor = transform(img).to(device)
    
    with torch.no_grad():
        prediction = model([img_tensor])[0]
        
    # Draw boxes
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(img)
    
    # Load Categories (Hardcoded or passed if needed, matching the dataset)
    # Ensure this matches your dataset's category mapping!
    # COCO usually starts classes at 1. 0 is background.
    
    for box, score, label in zip(prediction['boxes'], prediction['scores'], prediction['labels']):
        if score > threshold:
            box = box.cpu().numpy()
            draw.rectangle(box, outline="red", width=3)
            # You can add text label here if you have the map
            draw.text((box[0], box[1]), f"Cls: {label} ({score:.2f})", fill="red")
            
    img.save(save_path)
    print(f"Prediction saved to {save_path}")

# ----------------------------
# 6. Main Execution
# ----------------------------

def main():
    # --- Setup ---
    # Redirect print to file
    sys.stdout = DualLogger("training_log.txt")
    
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Using device: {device}")

    # PATHS - User Modify These!
    TRAIN_DIR = '/kaggle/input/project9/Casting.v10-v8.coco/train'
    TRAIN_ANN = '/kaggle/input/project9/Casting.v10-v8.coco/train/_annotations.coco.json'
    VALID_DIR = '/kaggle/input/project9/Casting.v10-v8.coco/valid'
    VALID_ANN = '/kaggle/input/project9/Casting.v10-v8.coco/valid/_annotations.coco.json'
    # Test directory can be used for visualization later
    
    # --- Dataset & Model ---
    print("Loading datasets...")
    train_dataset = CastingDataset(TRAIN_DIR, TRAIN_ANN, transforms=get_transform())
    valid_dataset = CastingDataset(VALID_DIR, VALID_ANN, transforms=get_transform())

    # DataLoaders
    train_data_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=4, shuffle=True, 
        num_workers=2, collate_fn=lambda x: tuple(zip(*x)))
    
    valid_data_loader = torch.utils.data.DataLoader(
        valid_dataset, batch_size=2, shuffle=False, 
        num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

    # Classes: 1 (background) + Number of actual categories
    # IMPORTANT: Check your dataset's largest ID. 
    # The model expects class indices 0..N.
    # In standard PyTorch FasterRCNN, 0 is background.
    # If your COCO IDs are 0, 1, 2... you might need to offset them by +1 in the dataset class 
    # OR ensure num_classes handles the max ID.
    # Let's inspect cat IDs dynamically:
    max_id = max(train_dataset.ids) if train_dataset.ids else 0
    # Actually we need max category ID
    cat_ids = train_dataset.coco.getCatIds()
    num_classes = len(cat_ids) + 1 # +1 to account for 0-indexing if 0 is used or background
    
    print(f"Number of classes (including background): {num_classes}")
    
    model = get_model_instance_segmentation(num_classes)
    model.to(device)

    # Optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    
    # Learning Rate Scheduler
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    # --- Training Loop ---
    num_epochs = 10 
    
    for epoch in range(num_epochs):
        train_one_epoch(model, optimizer, train_data_loader, device, epoch, print_freq=50)
        lr_scheduler.step()
        
        # Evaluate every epoch
        evaluate(model, valid_data_loader, device, num_classes, train_dataset.cat_id_to_name)
        
        # Save Output Weights
        torch.save(model.state_dict(), f"detector_weights_epoch_{epoch}.pth")

    print("Training Complete!")
    torch.save(model.state_dict(), "final_detector_weights.pth")

if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"An error occurred: {e}")
        # Normally you wouldn't suppress, but ensuring log file closes could be handled here


In [16]:
import torch
import os
import json
import numpy as np
from PIL import Image, ImageDraw, ImageFont
#from train_casting_detector import CastingDataset, get_model_instance_segmentation, get_transform, evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision.ops import box_iou

def visualize_test_results(model, dataset, device, num_images=10, threshold=0.5, output_dir="test_results"):
    model.eval()
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    print(f"\nGeneratng visualizations for {num_images} random test images...")
    
    indices = torch.randperm(len(dataset)).tolist()[:num_images]
    
    cat_id_to_name = dataset.cat_id_to_name
    
    # Define colors for different classes (optional, just random colors)
    colors = np.random.randint(0, 255, size=(100, 3), dtype=np.uint8)
    
    for i in indices:
        img, _ = dataset[i] # This img is a tensor because of the transform
        
        # We need the original image for drawing, or convert back from tensor
        # Easier to reload raw or convert
        # Let's convert back from tensor to PIL
        img_pil = Image.fromarray(img.mul(255).permute(1, 2, 0).byte().numpy())
        
        img_tensor = img.unsqueeze(0).to(device)
        
        with torch.no_grad():
            prediction = model(img_tensor)[0]
            
        draw = ImageDraw.Draw(img_pil)
        
        boxes = prediction['boxes'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()
        labels = prediction['labels'].cpu().numpy()
        
        has_detection = False
        for box, score, label in zip(boxes, scores, labels):
            if score > threshold:
                has_detection = True
                color = tuple(colors[label])
                draw.rectangle(box, outline=color, width=3)
                
                label_name = cat_id_to_name.get(label, str(label))
                text = f"{label_name}: {score:.2f}"
                
                # Draw text background
                text_size = draw.textbbox((0, 0), text) # Needs newer pillow, fallback if fail
                # simple fallback for older pillow:
                # draw.text((box[0], box[1]), text, fill=color)
                draw.rectangle([box[0], box[1] - 15, box[0] + 100, box[1]], fill=color)
                draw.text((box[0], box[1] - 15), text, fill="white")
        
        file_name = f"result_{i}.jpg"
        img_pil.save(os.path.join(output_dir, file_name))
        print(f"Saved {file_name}")

def calculate_detection_metrics(model, data_loader, device, num_classes, cat_id_to_name, iou_threshold=0.5):
    print("\n--- Calculating Detailed Metrics ---")
    model.eval()
    
    # Initialize confusion matrix and counts
    # Rows: Ground Truth, Cols: Prediction
    # We add 1 for "Background" (nothing detected where there was something, or detection where there was nothing)
    # However, standard object detection CM usually focuses on matched boxes. 
    # Let's track TP, FP, FN per class.
    
    stats = {cat_id: {'TP': 0, 'FP': 0, 'FN': 0} for cat_id in cat_id_to_name}
    
    # For Confusion Matrix:
    # We need to match predictions to GT.
    # If a pred matches a GT (IoU > thresh) and labels match -> Diagonal (TP)
    # If a pred matches a GT but labels don't match -> Misclassification
    # If a pred doesn't match any GT -> FP (Background predicted as Class)
    # If a GT isn't matched by any pred -> FN (Class predicted as Background)
    
    # Map category IDs to 0..N-1 indices for matrix
    sorted_cat_ids = sorted(cat_id_to_name.keys())
    cat_id_to_idx = {cid: i for i, cid in enumerate(sorted_cat_ids)}
    idx_to_cat_id = {i: cid for cid, i in cat_id_to_idx.items()}
    class_names = [cat_id_to_name[cid] for cid in sorted_cat_ids]
    
    # Extended classes for CM including Background
    # We will treat "Background" as the last index for visualization
    cm_num_classes = len(class_names) + 1 
    confusion_matrix = np.zeros((cm_num_classes, cm_num_classes), dtype=int)
    # Row: Actual, Col: Predicted
    # Last row/col is Background
    BG_IDX = len(class_names)
    
    y_true_all = []
    y_pred_all = []
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(img.to(device) for img in images)
            outputs = model(images)
            
            for target, output in zip(targets, outputs):
                gt_boxes = target['boxes'].to(device)
                gt_labels = target['labels'].to(device)
                
                pred_boxes = output['boxes'].to(device)
                pred_scores = output['scores'].to(device)
                pred_labels = output['labels'].to(device)
                
                # Filter predictions by score threshold (e.g. 0.5) before metric calc
                # This is standard to avoid counting very low confidence noise
                score_thresh = 0.5
                keep = pred_scores > score_thresh
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                
                if len(gt_boxes) == 0 and len(pred_boxes) == 0:
                    continue
                    
                # Match predictions to ground truth
                # If no GT, all preds are FP (Background -> Class)
                if len(gt_boxes) == 0:
                    for p_label in pred_labels:
                        pid = p_label.item()
                        if pid in stats: # metrics
                            stats[pid]['FP'] += 1
                        if pid in cat_id_to_idx: # CM
                            idx = cat_id_to_idx[pid]
                            confusion_matrix[BG_IDX, idx] += 1
                    continue
                
                # If no Preds, all GT are FN (Class -> Background)
                if len(pred_boxes) == 0:
                    for g_label in gt_labels:
                        gid = g_label.item()
                        if gid in stats:
                            stats[gid]['FN'] += 1
                        if gid in cat_id_to_idx:
                            idx = cat_id_to_idx[gid]
                            confusion_matrix[idx, BG_IDX] += 1
                    continue

                # Calculate IoU
                ious = box_iou(gt_boxes, pred_boxes)
                
                # For each GT, find best matching Pred
                # We need to ensure we don't double count. 
                # Basic greedy matching:
                
                gt_matched = set()
                pred_matched = set()
                
                # Iterate through all pairings, sorted by IoU desc
                # This is a simplified matching. COCO eval is more complex.
                
                # Flatten ious to (iou, gt_idx, pred_idx)
                iou_vals = []
                for g_i in range(len(gt_boxes)):
                    for p_i in range(len(pred_boxes)):
                        iou_vals.append((ious[g_i, p_i].item(), g_i, p_i))
                
                iou_vals.sort(key=lambda x: x[0], reverse=True)
                
                for iou, g_i, p_i in iou_vals:
                    if g_i in gt_matched or p_i in pred_matched:
                        continue
                        
                    if iou >= iou_threshold:
                        gt_matched.add(g_i)
                        pred_matched.add(p_i)
                        
                        g_lbl = gt_labels[g_i].item()
                        p_lbl = pred_labels[p_i].item()
                        
                        # Confusion Matrix update
                        if g_lbl in cat_id_to_idx and p_lbl in cat_id_to_idx:
                            g_idx = cat_id_to_idx[g_lbl]
                            p_idx = cat_id_to_idx[p_lbl]
                            confusion_matrix[g_idx, p_idx] += 1
                            
                        # Stats update
                        if g_lbl == p_lbl:
                            if g_lbl in stats:
                                stats[g_lbl]['TP'] += 1
                        else:
                            # Misclassification
                            # Technically this is a FN for g_lbl and FP for p_lbl
                            if g_lbl in stats: stats[g_lbl]['FN'] += 1
                            if p_lbl in stats: stats[p_lbl]['FP'] += 1
                            
                # Handle unmatched GT -> FN
                for g_i in range(len(gt_boxes)):
                    if g_i not in gt_matched:
                        g_lbl = gt_labels[g_i].item()
                        if g_lbl in stats: stats[g_lbl]['FN'] += 1
                        if g_lbl in cat_id_to_idx:
                            idx = cat_id_to_idx[g_lbl]
                            confusion_matrix[idx, BG_IDX] += 1
                            
                # Handle unmatched Preds -> FP
                for p_i in range(len(pred_boxes)):
                    if p_i not in pred_matched:
                        p_lbl = pred_labels[p_i].item()
                        if p_lbl in stats: stats[p_lbl]['FP'] += 1
                        if p_lbl in cat_id_to_idx:
                            idx = cat_id_to_idx[p_lbl]
                            confusion_matrix[BG_IDX, idx] += 1

    # --- Print Statistics ---
    print(f"\n{'Class':<20} {'TP':<6} {'FP':<6} {'FN':<6} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Acc':<10}")
    print("-" * 90)
    
    total_tp = 0
    total_fp = 0
    total_fn = 0
    
    for cat_id in sorted_cat_ids:
        s = stats[cat_id]
        tp, fp, fn = s['TP'], s['FP'], s['FN']
        
        total_tp += tp
        total_fp += fp
        total_fn += fn
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        # Accuracy per class (TP / (TP + FP + FN)) - often called Jaccard Index for detection
        acc = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        
        name = cat_id_to_name[cat_id]
        print(f"{name:<20} {tp:<6} {fp:<6} {fn:<6} {precision:<10.4f} {recall:<10.4f} {f1:<10.4f} {acc:<10.4f}")
        
    # Global Metrics (Micro-average)
    g_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    g_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    g_f1 = 2 * (g_precision * g_recall) / (g_precision + g_recall) if (g_precision + g_recall) > 0 else 0.0
    print("-" * 90)
    print(f"{'Global (Micro)':<20} {total_tp:<6} {total_fp:<6} {total_fn:<6} {g_precision:<10.4f} {g_recall:<10.4f} {g_f1:<10.4f}")
    
    # Save Confusion Matrix Plot
    plot_confusion_matrix(confusion_matrix, class_names + ["Background"])
    
    # Save text report
    with open("detailed_metrics_report.txt", "w") as f:
        f.write("Class-wise Metrics\n")
        f.write(f"{'Class':<20} {'TP':<6} {'FP':<6} {'FN':<6} {'Precision':<10} {'Recall':<10} {'F1-Score':<10}\n")
        for cat_id in sorted_cat_ids:
            s = stats[cat_id]
            tp, fp, fn = s['TP'], s['FP'], s['FN']
            p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0.0
            f.write(f"{cat_id_to_name[cat_id]:<20} {tp:<6} {fp:<6} {fn:<6} {p:<10.4f} {r:<10.4f} {f1:<10.4f}\n")
        
        f.write(f"\nGlobal F1: {g_f1:.4f}\n")
        f.write("Confusion Matrix Saved to confusion_matrix.png\n")

    return stats, confusion_matrix

def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png')
    print("Confusion Matrix plot saved to 'confusion_matrix.png'")
    plt.close()

def main():
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Using device: {device}")
    
    # 1. Configuration
    TEST_DIR = '/kaggle/input/project9/Casting.v10-v8.coco/test'
    TEST_ANN = '/kaggle/input/project9/Casting.v10-v8.coco/test/_annotations.coco.json'
    WEIGHTS_FILE = 'final_detector_weights.pth'
    
    if not os.path.exists(TEST_DIR):
        print(f"Test directory not found at {TEST_DIR}. Please check paths.")
        return

    # 2. Dataset
    print("Loading Test Dataset...")
    test_dataset = CastingDataset(TEST_DIR, TEST_ANN, transforms=get_transform())
    
    test_data_loader = torch.utils.data.DataLoader(
        test_dataset, batch_size=2, shuffle=False, 
        num_workers=2, collate_fn=lambda x: tuple(zip(*x)))
        
    # 3. Model
    # Determine num_classes same way as training
    cat_ids = test_dataset.coco.getCatIds()
    num_classes = len(cat_ids) + 1
    print(f"Num classes: {num_classes}")
    
    model = get_model_instance_segmentation(num_classes)
    
    if os.path.exists(WEIGHTS_FILE):
        print(f"Loading weights from {WEIGHTS_FILE}...")
        model.load_state_dict(torch.load(WEIGHTS_FILE, map_location=device))
    else:
        print(f"Weight file {WEIGHTS_FILE} not found! Cannot test.")
        return
        
    model.to(device)
    
    # 4. Evaluation (Metrics)
    evaluate(model, test_data_loader, device, num_classes, test_dataset.cat_id_to_name)
    
    # Rename the metrics file produced by evaluate to test_metrics
    if os.path.exists("metrics_results.json"):
        os.rename("metrics_results.json", "test_metrics_results.json")
        print("Renamed metrics output to 'test_metrics_results.json'")
        
    # Calculate detailed metrics
    calculate_detection_metrics(model, test_data_loader, device, num_classes, test_dataset.cat_id_to_name, iou_threshold=0.5)
        
    # 5. Visualization
    visualize_test_results(model, test_dataset, device, num_images=20, threshold=0.5)
    print("\nTesting Complete. check 'test_results' folder and 'test_metrics_results.json'.")

if __name__ == "__main__":
    main()
